3×3 Latin-Square Sudoku — Grover's Algorithm (Qiskit)
======================================================


In [1]:
"""
Demonstrates Grover's algorithm on a constrained combinatorial puzzle.
To keep simulation feasible (≤ 32 GB RAM), we fix the top row
as [0, 1, 2] and let Grover search the remaining 6 unknown cells.

Grid:
  +---+---+---+
  | 0 | 1 | 2 |   row 0  ← FIXED (classical, no qubits needed)
  +---+---+---+
  | v0| v1| v2|   row 1  ← quantum (2 bits each)
  +---+---+---+
  | v3| v4| v5|   row 2  ← quantum (2 bits each)
  +---+---+---+

Encoding : 2 bits per cell  (00=0, 01=1, 10=2, 11=invalid)
Variable qubits : 6 × 2 = 12
Clause ancillae : 9
Output qubit    : 1
Total           : 22 qubits  →  2^22 ≈ 64 MB  ✓ feasible

Fixed row  : [0, 1, 2]
Constraints involving unknown cells only:
  Row 1     : v0≠v1, v0≠v2, v1≠v2
  Row 2     : v3≠v4, v3≠v5, v4≠v5
  Col 0     : v0≠v3,  v0≠0,  v3≠0   → v0≠0, v3≠0, v0≠v3
  Col 1     : v1≠v4,  v1≠1,  v4≠1   → v1≠1, v4≠1, v1≠v4
  Col 2     : v2≠v5,  v2≠2,  v5≠2   → v2≠2, v5≠2, v2≠v5

Valid solutions (both rows must be permutations of {0,1,2}
  consistent with fixed row [0,1,2]):
  row1=[1,2,0], row2=[2,0,1]
  row1=[2,0,1], row2=[1,2,0]
  → M = 2 valid solutions

Search space : 4^6 = 4096
Optimal iters: ≈ π/4 · √(4096/2) ≈ 35
"""


"\nDemonstrates Grover's algorithm on a constrained combinatorial puzzle.\nTo keep simulation feasible (≤ 32 GB RAM), we fix the top row\nas [0, 1, 2] and let Grover search the remaining 6 unknown cells.\n\nGrid:\n  +---+---+---+\n  | 0 | 1 | 2 |   row 0  ← FIXED (classical, no qubits needed)\n  +---+---+---+\n  | v0| v1| v2|   row 1  ← quantum (2 bits each)\n  +---+---+---+\n  | v3| v4| v5|   row 2  ← quantum (2 bits each)\n  +---+---+---+\n\nEncoding : 2 bits per cell  (00=0, 01=1, 10=2, 11=invalid)\nVariable qubits : 6 × 2 = 12\nClause ancillae : 9\nOutput qubit    : 1\nTotal           : 22 qubits  →  2^22 ≈ 64 MB  ✓ feasible\n\nFixed row  : [0, 1, 2]\nConstraints involving unknown cells only:\n  Row 1     : v0≠v1, v0≠v2, v1≠v2\n  Row 2     : v3≠v4, v3≠v5, v4≠v5\n  Col 0     : v0≠v3,  v0≠0,  v3≠0   → v0≠0, v3≠0, v0≠v3\n  Col 1     : v1≠v4,  v1≠1,  v4≠1   → v1≠1, v4≠1, v1≠v4\n  Col 2     : v2≠v5,  v2≠2,  v5≠2   → v2≠2, v5≠2, v2≠v5\n\nValid solutions (both rows must be permutations of

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import math


# ── Constants ──────────────────────────────────────────────────
# Each unknown cell is encoded in 2 qubits: 00→0, 01→1, 10→2, 11→(invalid).
# Six unknown cells × 2 bits = 12 variable qubits total.
BITS       = 2
N_UNKNOWN  = 6          # v0..v5
N_VAR      = N_UNKNOWN * BITS   # 12

# The top row is treated as a classical constant — no qubits are allocated for it.
FIXED_ROW  = [0, 1, 2]  # top row is classical

# Cell map: v0=row1col0, v1=row1col1, v2=row1col2,
#           v3=row2col0, v4=row2col1, v5=row2col2

# ── Pair-inequality constraints (cell i ≠ cell j) ─────────────────────────────
# These generate the 9 clause ancilla qubits used in the oracle.
# Each tuple (i, j) means "the value encoded in variable i must differ
# from the value encoded in variable j."
PAIR_CLAUSES = [
    (0,1),(0,2),(1,2),   # row 1: all three cells mutually distinct
    (3,4),(3,5),(4,5),   # row 2: all three cells mutually distinct
    (0,3),(1,4),(2,5),   # columns: col pairs (v0≠v3, v1≠v4, v2≠v5)
]

# ── Constant-inequality constraints (cell i ≠ fixed value c) ──────────────────
# Derived from the fixed top row [0, 1, 2].
# Rather than allocating separate ancilla qubits, these are enforced in the
# oracle via X-gate pre/post-conditioning on the relevant variable qubits,
# folding the constant checks into the existing phase-kickback for free.
CONST_CLAUSES = [
    (0, 0),  # v0 ≠ 0  (col 0 has fixed 0)
    (1, 1),  # v1 ≠ 1  (col 1 has fixed 1)
    (2, 2),  # v2 ≠ 2  (col 2 has fixed 2)
    (3, 0),  # v3 ≠ 0  (col 0 has fixed 0)
    (4, 1),  # v4 ≠ 1  (col 1 has fixed 1)
    (5, 2),  # v5 ≠ 2  (col 2 has fixed 2)
]

# ── Qubit budget summary ───────────────────────────────────────────────────────
# Pair-clause ancillae : 9  (one per PAIR_CLAUSES entry)
# Const-clause ancillae: 6  (folded into oracle via X-gates — 0 extra qubits)
# Output (phase-kickback) qubit: 1
# ─────────────────────────────────────────────────────────────
# Total: 12 + 9 + 1 = 22 qubits  →  2^22 states ≈ 64 MB  ✓ feasible on laptop
N_PAIR_CLAUSE  = len(PAIR_CLAUSES)    # 9
N_CONST_CLAUSE = len(CONST_CLAUSES)   # 6  (but we fold into same ancilla bank)
N_CLAUSE       = N_PAIR_CLAUSE + N_CONST_CLAUSE   # 15... 




In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# HELPER: neq_pair
# Checks whether two 2-bit quantum variables encode different values.
#
# Circuit logic (self-inverse — call twice to uncompute):
#   1. XOR each bit of b with the corresponding bit of a  →  b[i] ^= a[i]
#      After this, b[i] == 0 iff a[i] == b[i], so b == |00⟩ iff a == b.
#   2. Flip all bits of b  →  b == |11⟩ iff a == b (i.e. |00⟩ → |11⟩).
#   3. Toffoli(b[0], b[1], ancilla)  →  ancilla flips iff b == |11⟩,
#      i.e. iff a == b.
#   4. X(ancilla)  →  ancilla = 1  iff  a ≠ b.
#   5. Unflip b, un-XOR b  →  restore b to its original value.
#
# Args:
#   qc        : QuantumCircuit to append gates to
#   a_qubits  : list of 2 qubits encoding value a (little-endian)
#   b_qubits  : list of 2 qubits encoding value b (little-endian)
#   ancilla   : single qubit; set to |1⟩ iff a ≠ b  (must start as |0⟩)
# ──────────────────────────────────────────────────────────────────────────────
def neq_pair(qc, a_qubits, b_qubits, ancilla):
    for i in range(BITS):
        qc.cx(a_qubits[i], b_qubits[i])   # XOR
    for i in range(BITS):
        qc.x(b_qubits[i])                 # flip
    qc.ccx(b_qubits[0], b_qubits[1], ancilla)  # Toffoli
    qc.x(ancilla)                              # ancilla = (a≠b)
    for i in range(BITS):
        qc.x(b_qubits[i])                 # unflip
    for i in range(BITS):
        qc.cx(a_qubits[i], b_qubits[i])   # un-XOR


# ──────────────────────────────────────────────────────────────────────────────
# HELPER: neq_const
# Checks whether a 2-bit quantum variable encodes a value different from a
# classical constant.
#
# Circuit logic (self-inverse — call twice to uncompute):
#   1. Identify which bits of const_val are 0.
#   2. Apply X to those qubit positions so that const_val maps to |11⟩.
#      (e.g. const_val=1 → binary 01 → flip bit 1 → cell==1 becomes |11⟩)
#   3. Toffoli → ancilla = 1  iff  cell == const_val (i.e. both bits are 1).
#   4. X(ancilla) → ancilla = 1  iff  cell ≠ const_val.
#   5. Unflip qubits to restore cell to its original encoding.
#
# No additional ancilla qubits are needed beyond the one passed in — this is
# the "free" constant-check technique that avoids extra clause ancillae.
#
# Args:
#   qc          : QuantumCircuit to append gates to
#   cell_qubits : list of 2 qubits encoding the cell value (little-endian)
#   const_val   : classical integer in {0, 1, 2} — the forbidden value
#   ancilla     : single qubit; set to |1⟩ iff cell ≠ const_val (must start |0⟩)
# ──────────────────────────────────────────────────────────────────────────────
def neq_const(qc, cell_qubits, const_val, ancilla):
    """
    ancilla = 1  iff  cell_qubits encodes a value ≠ const_val.

    Strategy: flip bits of cell so that const_val maps to |11⟩,
    then Toffoli → ancilla=1 means cell==const_val,
    X → ancilla=1 means cell≠const_val, then unflip.
    """
    # const_val in 2-bit binary (little-endian: bit0 first)
    bits = [(const_val >> i) & 1 for i in range(BITS)]

    # Flip qubits where const bit is 0, so const_val → |11⟩
    for i in range(BITS):
        if bits[i] == 0:
            qc.x(cell_qubits[i])

    qc.ccx(cell_qubits[0], cell_qubits[1], ancilla)  # ancilla=1 iff cell==const
    qc.x(ancilla)                                     # ancilla=1 iff cell≠const

    # Unflip
    for i in range(BITS):
        if bits[i] == 0:
            qc.x(cell_qubits[i])



In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# ORACLE: sudoku_oracle
#
# Marks valid Sudoku solutions with a phase flip via the output qubit.
# Implements the standard "compute → phase kickback → uncompute" pattern
# to keep all ancilla qubits clean (restored to |0⟩) after each call.
#
# Steps:
#   1. COMPUTE   — evaluate all 15 constraints into clause ancillae:
#                    · 9 pair-clauses  (cell i ≠ cell j)  via neq_pair
#                    · 6 const-clauses (cell i ≠ fixed value) via neq_const
#                  Each clause ancilla is set to |1⟩ iff its constraint is met.
#   2. KICK BACK — multi-controlled X on output_qubit (initialised as |−⟩)
#                  flips the phase of the full register iff ALL 15 ancillae
#                  are |1⟩, i.e. the candidate is a valid Latin-square solution.
#   3. UNCOMPUTE — apply every clause gate in reverse order to reset all
#                  clause ancillae back to |0⟩ (each helper is self-inverse).
#
# Args:
#   qc           : QuantumCircuit to append gates to
#   var_qubits   : 12-qubit register encoding v0…v5 (2 qubits per cell,
#                  little-endian, laid out as [v0[0], v0[1], v1[0], …])
#   clause_qubits: 15-qubit register for constraint ancillae
#                  (indices 0–8 → pair clauses, 9–14 → const clauses)
#   output_qubit : single qubit initialised to |−⟩ = H|1⟩ for phase kickback
# ──────────────────────────────────────────────────────────────────────────────
def sudoku_oracle(qc, var_qubits, clause_qubits, output_qubit):
    def cell(i):
        return var_qubits[i * BITS : i * BITS + BITS]

    # Forward: set all 15 clause ancillae
    for k, (i, j) in enumerate(PAIR_CLAUSES):
        neq_pair(qc, cell(i), cell(j), clause_qubits[k])

    for k, (ci, cv) in enumerate(CONST_CLAUSES):
        neq_const(qc, cell(ci), cv, clause_qubits[N_PAIR_CLAUSE + k])

    # Phase kickback when ALL 15 ancillae are |1⟩
    qc.mcx(list(clause_qubits), output_qubit)

    # Uncompute (reverse order)
    for k, (ci, cv) in reversed(list(enumerate(CONST_CLAUSES))):
        neq_const(qc, cell(ci), cv, clause_qubits[N_PAIR_CLAUSE + k])

    for k, (i, j) in reversed(list(enumerate(PAIR_CLAUSES))):
        neq_pair(qc, cell(i), cell(j), clause_qubits[k])


# ──────────────────────────────────────────────────────────────────────────────
# DIFFUSER: diffuser
#
# Implements the Grover diffuser (inversion-about-the-mean) tailored to the
# NON-UNIFORM initial superposition used here.
#
# Background:
#   A standard Hadamard diffuser assumes the initial state is the uniform
#   superposition |s⟩ = H^⊗n|0⟩. Here, each 2-qubit cell is prepared in a
#   uniform superposition over {0,1,2} only (3 states, not 4), so the initial
#   state is non-uniform in the computational basis. The diffuser must therefore
#   invert about THIS specific state, not the Hadamard basis.
#
# ──────────────────────────────────────────────────────────────────────────────
def diffuser(nqubits):
    """Diffuser matching the custom cell superposition state prep."""
    qc = QuantumCircuit(nqubits)

    # Uncompute state prep
    for i in range(N_UNKNOWN):
        cell_q = list(range(i*BITS, i*BITS+BITS))
        # Inverse of prepare_cell_superposition
        qc.x(cell_q[0])
        qc.ch(cell_q[0], cell_q[1])
        qc.x(cell_q[0])
        #theta = 2 * math.acos(1 / math.sqrt(3))
        theta = 2 * math.acos(math.sqrt(2/3))
        qc.ry(-theta, cell_q[0])

    # Flip all, multi-controlled Z, flip back
    qc.x(range(nqubits))
    qc.h(nqubits - 1)
    qc.mcx(list(range(nqubits - 1)), nqubits - 1)
    qc.h(nqubits - 1)
    qc.x(range(nqubits))

    # Reapply state prep
    for i in range(N_UNKNOWN):
        cell_q = list(range(i*BITS, i*BITS+BITS))
        #theta = 2 * math.acos(1 / math.sqrt(3))
        theta = 2 * math.acos(math.sqrt(2/3))
        qc.ry(theta, cell_q[0])
        qc.x(cell_q[0])
        qc.ch(cell_q[0], cell_q[1])
        qc.x(cell_q[0])

    return qc



In [5]:
# Prepare an equal superposition of |00⟩, |01⟩, |10⟩ (amplitide 1/√3 each).
# The invalid codeword |11⟩ is left with zero amplitude, restricting each
# cell's search space to the three valid symbols {0, 1, 2}.
def prepare_cell_superposition(qc, cell_qubits):

    # Ry(2*arccos(1/sqrt(3))) on first qubit → amplitude split 1/3 : 2/3
    
    theta = 2 * math.acos(math.sqrt(2/3))
    qc.ry(theta, cell_qubits[0])
    # Hadamard on second qubit controlled on first being |0⟩
    qc.x(cell_qubits[0])
    qc.ch(cell_qubits[0], cell_qubits[1])
    qc.x(cell_qubits[0])

In [6]:
# ──────────────────────────────────────────────────────────────
# Build circuit
# Assemble the full Grover circuit for the Sudoku search.

# Structure:
#     1. State prep  — each cell initialised to (1/√3)(|00⟩+|01⟩+|10⟩)
#     2. Output qubit set to |−⟩ = X·H|0⟩ for phase kickback
#     3. num_iterations × (Oracle + Diffuser)
#     4. Measure all 12 variable qubits
# ──────────────────────────────────────────────────────────────    
def build_circuit(num_iterations):
    var_reg    = QuantumRegister(N_VAR,    name='v')
    clause_reg = QuantumRegister(N_CLAUSE, name='c')
    out_reg    = QuantumRegister(1,        name='out')
    meas_reg   = ClassicalRegister(N_VAR,  name='meas')

    qc = QuantumCircuit(var_reg, clause_reg, out_reg, meas_reg)

    # Prepare each cell in superposition of {0,1,2} only — no |11⟩
    for i in range(N_UNKNOWN):
        prepare_cell_superposition(qc, var_reg[i*BITS : i*BITS+BITS])

    qc.x(out_reg)
    qc.h(out_reg)
    qc.barrier()
    
    for it in range(num_iterations):
        sudoku_oracle(qc, var_reg, clause_reg, out_reg[0])
        qc.barrier()
        qc.append(diffuser(N_VAR).to_gate(label='Diffuser'), var_reg)
        qc.barrier()
        print(f"  Iteration {it+1}/{num_iterations} done")

    qc.measure(var_reg, meas_reg)
    return qc


# Convert a measured bitstring to a list of 6 cell values [v0…v5].
# Qiskit returns bitstrings in big-endian order (rightmost bit = qubit 0),
# so we reverse the full string first, then reverse each 2-bit chunk to
# recover the little-endian encoding used during circuit construction.
def decode(bitstring):
    bits = bitstring[::-1]  # reverse whole string
    return [int(bits[i*BITS:(i+1)*BITS][::-1], 2) for i in range(N_UNKNOWN)]  # reverse each chunk


# Return True if the bitstring encodes a valid Latin-square solution.
def verify(bitstring):
    v = decode(bitstring)
    if any(x > 2 for x in v):   # ← add this line
        return False
    pairs_ok = all(v[i] != v[j] for i, j in PAIR_CLAUSES)
    const_ok = all(v[ci] != cv for ci, cv in CONST_CLAUSES)
    return pairs_ok and const_ok


# Print the full 3×3 grid with the fixed top row and decoded rows 1 & 2.
def print_grid(values, title=""):
    """Print full 3×3 grid (fixed top row + decoded rows 1 & 2)."""
    sep = "  +---+---+---+"
    if title:
        print(f"\n  {title}")
    print(sep)
    # Fixed row 0
    print("  | " + " | ".join(str(x) for x in FIXED_ROW) + " |  ← fixed")
    print(sep)
    # Row 1: v0, v1, v2
    print("  | " + " | ".join(str(values[i]) for i in range(3)) + " |")
    print(sep)
    # Row 2: v3, v4, v5
    print("  | " + " | ".join(str(values[i]) for i in range(3, 6)) + " |")
    print(sep)



In [7]:
# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────

# ── Grover parameters ──────────────────────────────────────────────────────────
# Optimal iteration count: k ≈ (π/4)·√(N/M)
# N = 3^6 = 729 (valid codewords per cell × 6 cells)
# M = 2 solutions: row1=[1,2,0],row2=[2,0,1] and row1=[2,0,1],row2=[1,2,0]
N_STATES       = 3 ** N_UNKNOWN      # 729
M_SOLS         = 2                   # [1,2,0,2,0,1] and [2,0,1,1,2,0]
OPT_ITERS      = max(1, round((math.pi / 4) * math.sqrt(N_STATES / M_SOLS)))
NUM_ITERATIONS = OPT_ITERS           
SHOTS          = 2048

# ── Summary banner ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  3×3 Latin-Square Sudoku — Grover's Algorithm")
print("  (Top row fixed as [0,1,2] for feasible simulation)")
print("=" * 60)
print(f"  Encoding        : 2 bits/cell, values {{0,1,2}}")
print(f"  Unknown cells   : {N_UNKNOWN}  (rows 1 & 2)")
print(f"  Variable qubits : {N_VAR}")
print(f"  Clause ancillae : {N_CLAUSE}")
print(f"  Total qubits    : {N_VAR + N_CLAUSE + 1}  "
      f"(2^{N_VAR+N_CLAUSE+1} = {2**(N_VAR+N_CLAUSE+1):,} amplitudes ✓)")
print(f"  Search space    : 4^{N_UNKNOWN} = {N_STATES:,}")
print(f"  Valid solutions : {M_SOLS}")
print(f"  Optimal iters   : {OPT_ITERS}")
print(f"  Shots           : {SHOTS}")

# ── Build ──────────────────────────────────────────────────────────────────────
print("\n  Building circuit...")
qc = build_circuit(num_iterations=NUM_ITERATIONS)

# Transpile to a small universal gate set for realistic depth/gate estimates.
# optimization_level=0 avoids gate cancellations that would obscure the raw structure.
qc_t = transpile(qc,
                  basis_gates=['cx', 'u', 'x', 'h', 'ccx'],
                  optimization_level=0)
print(f"  Total qubits    : {qc.num_qubits}")
print(f"  Circuit depth   : {qc_t.depth()}")
print(f"  Gate count      : {sum(qc_t.count_ops().values())}")

# ── Simulate ───────────────────────────────────────────────────────────────────
# Statevector simulation is exact (no shot noise beyond sampling).
# Results are sorted descending so the two solution bitstrings appear first.
print(f"\n  Simulating ({SHOTS} shots)...")
sim    = AerSimulator(method='statevector')
result = sim.run(qc_t, shots=SHOTS).result()
counts = result.get_counts()

sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)


  3×3 Latin-Square Sudoku — Grover's Algorithm
  (Top row fixed as [0,1,2] for feasible simulation)
  Encoding        : 2 bits/cell, values {0,1,2}
  Unknown cells   : 6  (rows 1 & 2)
  Variable qubits : 12
  Clause ancillae : 15
  Total qubits    : 28  (2^28 = 268,435,456 amplitudes ✓)
  Search space    : 4^6 = 729
  Valid solutions : 2
  Optimal iters   : 15
  Shots           : 2048

  Building circuit...
  Iteration 1/15 done
  Iteration 2/15 done
  Iteration 3/15 done
  Iteration 4/15 done
  Iteration 5/15 done
  Iteration 6/15 done
  Iteration 7/15 done
  Iteration 8/15 done
  Iteration 9/15 done
  Iteration 10/15 done
  Iteration 11/15 done
  Iteration 12/15 done
  Iteration 13/15 done
  Iteration 14/15 done
  Iteration 15/15 done
  Total qubits    : 28
  Circuit depth   : 42263
  Gate count      : 50610

  Simulating (2048 shots)...


In [8]:
# ── Top-10 results ─────────────────────────────────────────────────────────────
# Show the highest-count bitstrings with their decoded grids.
# A valid solution must pass all pair and constant constraints (see verify()).
print(f"\n{'─'*60}")
print("  Top 10 Results   (✓ = valid solution)")
print(f"{'─'*60}")
for rank, (bitstr, cnt) in enumerate(sorted_counts[:10], 1):
    v     = decode(bitstr)
    valid = verify(bitstr)
    tag   = "✓ VALID" if valid else "✗ invalid"
    print(f"\n  Rank {rank:>2}  |  Count: {cnt:>4}/{SHOTS}  |  {tag}")
    print_grid(v)

# ── Aggregate statistics ────────────────────────────────────────────────────────
# valid_shots  — total shots that collapsed onto a valid solution bitstring
# unique_valid — distinct valid bitstrings (expect exactly 2 for this puzzle)
valid_shots  = sum(c for b, c in counts.items() if verify(b))
unique_valid = [b for b in counts if verify(b)]

print(f"\n{'─'*60}")
print("  Summary")
print(f"{'─'*60}")
print(f"  Unique outcomes   : {len(counts)}")
print(f"  Unique valid grids: {len(unique_valid)}  (expect 2)")
print(f"  Shots on valid    : {valid_shots}/{SHOTS} "
        f"({100 * valid_shots / SHOTS:.1f}%)")
print(f"  Shots on invalid  : {SHOTS-valid_shots}/{SHOTS} "
        f"({100*(SHOTS-valid_shots)/SHOTS:.1f}%)")

# ── Valid grids ─────────────────────────────────────────────────────────────────
# Print each discovered solution as a full 3×3 grid with its shot count.
# High shot concentration on exactly 2 grids confirms Grover amplification worked.
print(f"\n{'─'*60}")
print("  Valid Sudoku Grids Found")
print(f"{'─'*60}")
for b in unique_valid:
    print_grid(decode(b), title=f"shots = {counts[b]}")



────────────────────────────────────────────────────────────
  Top 10 Results   (✓ = valid solution)
────────────────────────────────────────────────────────────

  Rank  1  |  Count: 1073/2048  |  ✓ VALID
  +---+---+---+
  | 0 | 1 | 2 |  ← fixed
  +---+---+---+
  | 2 | 0 | 1 |
  +---+---+---+
  | 1 | 2 | 0 |
  +---+---+---+

  Rank  2  |  Count:  975/2048  |  ✓ VALID
  +---+---+---+
  | 0 | 1 | 2 |  ← fixed
  +---+---+---+
  | 1 | 2 | 0 |
  +---+---+---+
  | 2 | 0 | 1 |
  +---+---+---+

────────────────────────────────────────────────────────────
  Summary
────────────────────────────────────────────────────────────
  Unique outcomes   : 2
  Unique valid grids: 2  (expect 2)
  Shots on valid    : 2048/2048 (100.0%)
  Shots on invalid  : 0/2048 (0.0%)

────────────────────────────────────────────────────────────
  Valid Sudoku Grids Found
────────────────────────────────────────────────────────────

  shots = 975
  +---+---+---+
  | 0 | 1 | 2 |  ← fixed
  +---+---+---+
  | 1 | 2 | 0 |

In [9]:
# ── Histogram of top-20 outcomes ───────────────────────────────────────────────
# Each bar represents one measured bitstring, labelled as its full 3×3 grid.
# Green = valid Latin-square solution; blue = invalid (noise / non-solution states).
# Grover amplification should concentrate shots onto the 2 green bars.
top20      = dict(sorted_counts[:20])
bar_colors = ['#2ecc71' if verify(b) else '#3498db' for b in top20]

def bar_label(b):
    v = decode(b)
    return (f"{FIXED_ROW[0]} {FIXED_ROW[1]} {FIXED_ROW[2]}\n"
            f"{v[0]} {v[1]} {v[2]}\n"
            f"{v[3]} {v[4]} {v[5]}")

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(range(len(top20)), list(top20.values()), color=bar_colors)
ax.set_xticks(range(len(top20)))
ax.set_xticklabels([bar_label(b) for b in top20],
                    fontsize=7.5, family='monospace')
ax.set_ylabel("Shot count")
ax.set_xlabel("Full 3×3 grid (top row fixed)")
ax.set_title(
    f"3×3 Latin-Square Sudoku — Grover's Algorithm\n"
    f"Top row fixed=[0,1,2] | 2 bits/cell | 22 qubits | "
    f"{NUM_ITERATIONS} iters | {SHOTS} shots"
)
ax.legend(handles=[
    Patch(color='#2ecc71', label='Valid Latin square ✓'),
    Patch(color='#3498db', label='Invalid'),
])
plt.tight_layout()

# Display inline (notebook) and save to disk for the GitHub render.
from IPython.display import display
display(fig)
#plt.close()
out_img = "grover_3x3_sudoku_final.png"
plt.savefig(out_img, dpi=150)
plt.show()
print(f"\n  Histogram saved → {out_img}")
print("\n  Done.")

<Figure size 1600x600 with 1 Axes>


  Histogram saved → grover_3x3_sudoku_final.png

  Done.


C:\Users\Administrator\AppData\Local\Temp\ipykernel_46188\64342275.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
print(N_CLAUSE)  # should print 15
print(N_PAIR_CLAUSE)    # should print 9
print(N_CONST_CLAUSE)   # should print 6
print(OPT_ITERS)

15
9
6
15
